# Pair Target Encoding (expanding window)

Генерація `train_pair_features` та `test_pair_features`:
- Train: expanding window по тижнях (тиждень N маппиться по історії тижнів 1..N-1)
- Test: маппиться по всьому train

In [ ]:
import pandas as pd
import numpy as np
import gc
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = 'int-20-h-2026-final-task/'

In [ ]:
train = pd.read_csv(DATA_PATH + 'train_data_fixed.csv', parse_dates=['order_created_at'])
test = pd.read_csv(DATA_PATH + 'test.csv', parse_dates=['order_created_at'])

# Фікс platform
platform_replacements = {'App': 'APP', 'web': 'WEB', 'MOD': 'MOB', 'Web': 'WEB'}
train['platform'] = train['platform'].replace(platform_replacements)
test['platform'] = test['platform'].replace(platform_replacements)

print(f'Train: {train.shape}')
print(f'Test:  {test.shape}')

In [ ]:
# Тижні
train['week'] = train['order_created_at'].dt.isocalendar().week.astype(int)
weeks = sorted(train['week'].unique())
print(f'Тижні: {weeks}')

In [ ]:
# Пари категоріальних колонок
cat_cols = ['psp_id', 'cardbrand', 'cardtype', 'currency', 'platform',
            'payment_source', 'transaction_type', 'merchant_token_type',
            'is_secured', 'card_pan_type', 'mcc_id']

selected_pairs = list(combinations(cat_cols, 2))
print(f'Пар: {len(selected_pairs)}')

In [ ]:
def calc_smoothed_ar_dict(history_df, cols, global_ar, alpha=20):
    stats = history_df.groupby(cols)['is_success'].agg(['sum', 'count'])
    ar = stats['sum'] / stats['count']
    smoothed = (stats['count'] * ar + alpha * global_ar) / (stats['count'] + alpha)
    return smoothed.to_dict(), stats['count'].to_dict()

GLOBAL_AR = train['is_success'].mean()
ALPHA = 20
BATCH_SIZE = 15

In [ ]:
%%time
# Expanding window target encoding для train — батчами
week_indices = {w: train.index[train['week'] == w].values for w in weeks}

for batch_start in range(0, len(selected_pairs), BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, len(selected_pairs))
    batch_pairs = list(enumerate(selected_pairs[batch_start:batch_end], start=batch_start))

    batch_ar = {idx: np.full(len(train), np.nan, dtype=np.float32) for idx, _ in batch_pairs}
    batch_cnt = {idx: np.zeros(len(train), dtype=np.int32) for idx, _ in batch_pairs}

    for wi, current_week in enumerate(weeks):
        if wi == 0:
            continue
        history_idx = np.concatenate([week_indices[w] for w in weeks[:wi]])
        history = train.iloc[history_idx]
        current_idx = week_indices[current_week]

        for idx, pair in batch_pairs:
            cols = list(pair)
            ar_dict, cnt_dict = calc_smoothed_ar_dict(history, cols, GLOBAL_AR, ALPHA)
            keys = list(zip(*(train.loc[current_idx, c] for c in cols)))
            batch_ar[idx][current_idx] = np.array([ar_dict.get(k, np.nan) for k in keys], dtype=np.float32)
            batch_cnt[idx][current_idx] = np.array([cnt_dict.get(k, 0) for k in keys], dtype=np.int32)

    for idx, _ in batch_pairs:
        train[f'pair_{idx}_ar'] = batch_ar[idx]
        train[f'pair_{idx}_count'] = batch_cnt[idx]

    del batch_ar, batch_cnt
    gc.collect()
    print(f'Batch {batch_start}-{batch_end-1} done ({batch_end}/{len(selected_pairs)})')

print(f'\nDone! {len(selected_pairs)} пар')

In [ ]:
# Відрізаємо перший тиждень (немає історії для encoding)
cutoff = train['order_created_at'].min() + pd.Timedelta(weeks=1)
train = train[train['order_created_at'] >= cutoff].reset_index(drop=True)
print(f'Train після cutoff: {train.shape}')

In [ ]:
# Зберігаємо train pair features
pair_cols = [c for c in train.columns if c.startswith('pair_')]
train_save = train[['psp_order_id'] + pair_cols]
train_save.to_parquet(DATA_PATH + 'train_pair_features.parquet', engine='pyarrow', index=False)
print(f'Збережено train: {train_save.shape}')
del train_save
gc.collect()

In [ ]:
%%time
# Target encoding для test — по всьому train
for idx, pair in enumerate(selected_pairs):
    cols = list(pair)
    ar_dict, cnt_dict = calc_smoothed_ar_dict(train, cols, GLOBAL_AR, ALPHA)
    keys = list(zip(*(test[c] for c in cols)))
    test[f'pair_{idx}_ar'] = np.array([ar_dict.get(k, np.nan) for k in keys], dtype=np.float32)
    test[f'pair_{idx}_count'] = np.array([cnt_dict.get(k, 0) for k in keys], dtype=np.int32)

print(f'Test coverage (pair 0): {test["pair_0_ar"].notna().mean():.2%}')

In [ ]:
# Зберігаємо test pair features
pair_cols = [c for c in test.columns if c.startswith('pair_')]
test_save = test[['psp_order_id'] + pair_cols]
test_save.to_parquet(DATA_PATH + 'test_pair_features.parquet', engine='pyarrow', index=False)
print(f'Збережено test: {test_save.shape}')
del test_save
gc.collect()